RUNTIME: H100 (recommended), A100 fallback, T4 only if necessary
INSTRUCTIONS:

1. Set runtime to GPU: Runtime → Change runtime type → GPU → H100 (or A100 if H100 unavailable)
2. Run this notebook from inside the cloned repo (e.g., `/content/<repo>/notebooks`) so `src/` imports work
3. Ensure repo data files exist in `data/` (train/val/test + fraction files from `00_eda`)
4. Use repo paths (`data/`, `results/`) instead of ad-hoc temp paths for reproducibility
5. Architect runs for crash safety: run in small chunks, save after each chunk, and resume from saved CSVs/checkpoints
6. Save notebook state regularly (Ctrl+S) and keep checkpoints in Drive when training is long


In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────
!pip install -q -U "transformers<5" accelerate datasets

# ── 2. Optional Google Drive mount ────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"
except Exception:
    SAVE_DIR = None

# ── 3. Standard imports ───────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── 4. Resolve repo/data/results paths ────────────────────────────
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src").exists() and (candidate / "data").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate repo root with src/ and data/ directories")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / "data"
RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── 5. Sanity checks ──────────────────────────────────────────────
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)


Mounted at /content/drive
PyTorch: 2.10.0+cu128
GPU available: True
GPU: NVIDIA A100-SXM4-40GB
